# Generalized Knowledge Distillation Trainer
Link: 
- https://huggingface.co/docs/trl/en/gkd_trainer

The key aspects of GKD are:

- It addresses the train-inference distribution mismatch in auto-regressive sequence models by training the student model on its self-generated output sequences.
- GKD allows flexibility in choosing different divergence measures between student and teacher models via the generalized Jensen-Shannon Divergence (JSD), which can be useful when the student lacks the capacity to fully mimic the teacher.

## Qwen2

In [1]:
# sudo apt install python3.12-dev

In [2]:
import re

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl.experimental.gkd import GKDConfig, GKDTrainer

/home/imaha/Desktop/study/small-LLM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_605683/2544254765.py:6: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.gkd import GKDConfig, GKDTrainer


In [3]:
SEED = 67
N_TRAIN = 2000
N_VAL = 200

In [4]:
ds = load_dataset(
    "csebuetnlp/xlsum",
    data_dir="ukrainian",
    revision="refs/convert/parquet",
)

train_raw = ds["train"].shuffle(seed=SEED).select(range(N_TRAIN))
val_raw = ds["validation"].shuffle(seed=SEED).select(range(N_VAL))
print(len(train_raw), len(val_raw))

2000 200


In [5]:
def xlsum_prompt(record, max_chars=1000):
    title = re.sub(r" {2,}", " ", record["title"])
    text = re.sub(r" {2,}", " ", record["text"])[:max_chars]
    return (
        "Підсумуй цю статтю:\n "
        f"Назва: {title}\n "
        # f"Посилання: {record['url']}\n"
        f"Текст статті: \n{text}\n "
    )
def to_messages(record):
    return {
        "messages": [
            {"role": "user", "content": xlsum_prompt(record)},
            {"role": "assistant", "content": record["summary"]},
        ]
    }

train_dataset = train_raw.map(to_messages, remove_columns=train_raw.column_names)
eval_dataset = val_raw.map(to_messages, remove_columns=val_raw.column_names)
train_dataset[0]["messages"][0]["content"][:400]

'Підсумуй цю статтю:\n Назва: О.Мороз закликає протистояти шахрайству влади\n Текст статті: \nУ листі до лідера "Нашої України" та провідника комуністів, Олександр Мороз звертає увагу на прояви того, що він називає "масштабним чорним піаром" вже на початковому етапі передвиборчої кампанії. Олександр Мороз не виключає, що це може перетворити виборчу кампанію - цитата - "на один з найбрудніших періодів '

In [6]:
student_id = "Qwen/Qwen2-0.5B-Instruct"
teacher_id = "Qwen/Qwen2-1.5B-Instruct"

use_cuda = torch.cuda.is_available()
bf16 = use_cuda and torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if bf16 else (torch.float16 if use_cuda else torch.float32)


In [7]:
tokenizer = AutoTokenizer.from_pretrained(student_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# The model to optimise
model = AutoModelForCausalLM.from_pretrained(student_id, dtype=dtype)
# The teacher model to calculate the KL divergence against
teacher_model = AutoModelForCausalLM.from_pretrained(teacher_id, dtype=dtype)

teacher_model.save_pretrained("../models/qwen2-1.5b-teacher")
tokenizer.save_pretrained("../models/qwen2-1.5b-teacher")

def clip_messages(row, max_prompt_tokens=448, max_new_tokens=64):
    user = row["messages"][0]["content"]
    summary = row["messages"][1]["content"]
    max_chars = len(user)
    while True:
        messages = [{"role": "user", "content": user[:max_chars]}]
        prompt = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )
        n_tokens = len(tokenizer(prompt, add_special_tokens=False)["input_ids"])
        if n_tokens <= max_prompt_tokens or max_chars <= 1:
            break
        next_chars = max(1, int(max_chars * max_prompt_tokens / n_tokens))
        if next_chars >= max_chars:
            next_chars = max_chars - 1
        max_chars = next_chars
    summary_ids = tokenizer(
        summary, add_special_tokens=False, truncation=True, max_length=max_new_tokens
    )["input_ids"]
    return {
        "messages": [
            {"role": "user", "content": user[:max_chars]},
            {
                "role": "assistant",
                "content": tokenizer.decode(summary_ids, skip_special_tokens=True).strip(),
            },
        ]
    }

train_dataset = train_dataset.map(clip_messages)
eval_dataset = eval_dataset.map(clip_messages)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.11it/s]


In [8]:
training_args = GKDConfig(
    output_dir="../models/gkd-qwen2-0.5b-xlsum",
    per_device_train_batch_size=1,
    per_device_eval_batch_size = 1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-5,
    num_train_epochs=1,
    max_length=512,
    lmbda=1.0,
    beta=0.5,
    temperature=0.9,
    max_new_tokens=64,
    seq_kd=False,
    logging_steps=10,
    eval_strategy="no",
    save_strategy="epoch",
    bf16=bf16,
    fp16=use_cuda and not bf16,
    seed=SEED,
    report_to="none",
)

trainer = GKDTrainer(
    model=model,
    teacher_model=teacher_model,
    args=training_args,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)
trainer.generation_config.top_k = 50
trainer.train()
trainer.save_model(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.
[transformers] Passing `generation_config` together with generation-related arguments=({'return_dict_in_generate'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss
10,0.136801
20,0.135508
30,0.133803
40,0.122091
50,0.117013
60,0.120911
70,0.119119
80,0.110380
90,0.115334
100,0.118086


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.86it/s]


('../models/gkd-qwen2-0.5b-xlsum/tokenizer_config.json',
 '../models/gkd-qwen2-0.5b-xlsum/chat_template.jinja',
 '../models/gkd-qwen2-0.5b-xlsum/tokenizer.json')

In [ ]:
import torch.nn.functional as F

def completion_nll(model, dataset):
    total_nll, n_tokens = 0.0, 0
    model.eval()
    for row in dataset:
        batch = trainer.data_collator([row])
        tensors = {
            key: value.to(model.device)
            for key, value in batch.items()
            if torch.is_tensor(value)
        }
        labels = tensors["labels"]
        with torch.no_grad():
            logits = model(
                input_ids=tensors["input_ids"],
                attention_mask=tensors["attention_mask"],
            ).logits[:, :-1, :]
        shift_labels = labels[:, 1:]
        mask = shift_labels != -100
        n = int(mask.sum())
        if n == 0:
            continue
        total_nll += F.cross_entropy(
            logits[mask].float(), shift_labels[mask], reduction="sum"
        ).item()
        n_tokens += n
    return total_nll / n_tokens

teacher_model.to(model.device)
print("teacher cross-entropy", completion_nll(teacher_model, eval_dataset))
teacher_model.to("cpu")
torch.cuda.empty_cache()
print("student cross-entropy", completion_nll(model, eval_dataset))


In [9]:
torch.cuda.empty_cache()
print(trainer.evaluate()["eval_loss"]) # Jensen–Shannon

Training Loss,Validation Loss,Step
0.104345,0.128785,250


0.12878461182117462


### Rouge_L score

In [10]:
from rouge_score import rouge_scorer

In [11]:
N_TEST = 100
test_raw = ds["test"].shuffle(seed=SEED).select(range(N_TEST))
records = list(test_raw)

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

In [12]:
def prompt_inputs(record, max_prompt_tokens=448):
    max_chars = 1000
    inputs = None
    while True:
        messages = [{"role": "user", "content": xlsum_prompt(record, max_chars=max_chars)}]
        prompt = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
        n_tokens = inputs["input_ids"].shape[1]
        if n_tokens <= max_prompt_tokens or max_chars <= 1:
            break
        # Shrink the article and keep the chat template, including the assistant header.
        next_chars = max(1, int(max_chars * max_prompt_tokens / n_tokens))
        if next_chars >= max_chars:
            next_chars = max_chars - 1
        max_chars = next_chars
    return inputs

def reference_summary(record, max_tokens=64):
    ids = tokenizer(
        record["summary"],
        add_special_tokens=False,
        truncation=True,
        max_length=max_tokens,
    )["input_ids"]
    return tokenizer.decode(ids, skip_special_tokens=True).strip()

def generate_summary(model, record, max_new_tokens=64, max_prompt_tokens=448):
    inputs = prompt_inputs(record, max_prompt_tokens=max_prompt_tokens)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy decoding
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=model.generation_config.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def rouge_on_xlsum(model, records):
    totals = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    model.eval()
    for record in records:
        prediction = generate_summary(model, record)
        scores = scorer.score(reference_summary(record), prediction)
        for key in totals:
            totals[key] += scores[key].fmeasure
    return {key: value / len(records) for key, value in totals.items()}

def prepare(model):
    return model.to(device)

In [13]:
rouge_student = rouge_on_xlsum(prepare(model), records)
print("distilled", rouge_student)
model.to("cpu")

rouge_teacher = rouge_on_xlsum(prepare(teacher_model), records)
print("teacher", rouge_teacher)
teacher_model.to("cpu")

distilled {'rouge1': 0.05333333333333333, 'rouge2': 0.0, 'rougeL': 0.05333333333333333}
teacher {'rouge1': 0.055, 'rouge2': 0.01, 'rougeL': 0.055}


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [14]:
base = AutoModelForCausalLM.from_pretrained(student_id, dtype=dtype)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("base", rouge_on_xlsum(prepare(base), records))
base.to("cpu")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 13186.06it/s]


base {'rouge1': 0.04657142857142857, 'rouge2': 0.01966666666666667, 'rougeL': 0.04657142857142857}


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [15]:
print("gold:", reference_summary(records[0]))
print("distilled:", generate_summary(prepare(model), records[0]))

gold: Як показують дані нового дослідження, щоденна доза комбінованої таблетки з чотирьох препаратів може скоротити кількість інфарктів
distilled: Ця стаття повідає про нову і дешеву таблетку, яка відповідає на проблеми серця та інсультів. Цей продукт вже відкриває відносину


In [16]:
for key in rouge_teacher:
    print(key, rouge_student[key] - rouge_teacher[key])

rouge1 -0.0016666666666666705
rouge2 -0.01
rougeL -0.0016666666666666705


In [17]:
def param_count(model):
    return sum(p.numel() for p in model.parameters())

student_n = param_count(model)
teacher_n = param_count(teacher_model)

print(f"student: {student_n / 1e6:.1f} M")
print(f"teacher: {teacher_n / 1e6:.1f} M")
print(f"compression: {teacher_n / student_n:.2f}x")

student: 494.0 M
teacher: 1543.7 M
compression: 3.12x


In [18]:
def weight_mb(model):
    nbytes = sum(p.numel() * p.element_size() for p in model.parameters())
    return nbytes / 1024**2

print(f"student: {weight_mb(model):.0f} MB")
print(f"teacher: {weight_mb(teacher_model):.0f} MB")

student: 942 MB
teacher: 2944 MB
